# Guided Scenario Runner

This notebook imports the shared simulation package, discovers every official case study, runs each A/B pair, and visualises the results.

The workflow stays aligned with the dashboard and API because every run comes from the same `case_study.json` manifests and preset library.


## Setup And Environment Check

This section prepares the notebook environment, locates the repository root, loads helper functions, and discovers the official case studies without requiring optional plotting libraries to be installed.

If you still see an older saved Plotly traceback under the setup cell, rerun that cell once to refresh the notebook output.


In [13]:
from __future__ import annotations

from collections import Counter
from html import escape
from importlib import import_module
from pathlib import Path
import sys

from IPython.display import HTML, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "code" / "restaurant_simulation").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root from this notebook.")


def table_column_style(column: str) -> str:
    if column in {"description", "message"}:
        return "width: 34%; min-width: 38ch;"
    if column in {"title", "scenario", "focus", "option_label", "metric"}:
        return "width: 22%; min-width: 24ch;"
    if column in {
        "restaurant_layout",
        "queue_structure",
        "reservation_policy",
        "seating_policy",
        "service_policy",
        "arrival_scenario",
        "bands",
        "capacity_mix",
    }:
        return "width: 20%; min-width: 22ch;"
    if column in {
        "average_wait_time",
        "table_utilization_overall",
        "service_level_within_15_min",
        "max_queue_length",
        "peak_time",
    }:
        return "width: 14ch; min-width: 16ch;"
    if column in {"id", "case_study", "pair", "version", "time", "group", "table"}:
        return "width: 9ch; min-width: 9ch;"
    if column.endswith("_enabled") or column in {
        "tables",
        "total_seats",
        "largest_table",
        "default_reset",
        "arrival_groups",
        "max_party_size",
        "reservations",
        "walkins",
        "hold_minutes",
        "peak_waiting",
        "final_waiting",
        "snapshots",
        "groups_served",
        "groups_abandoned",
    }:
        return "width: 12ch; min-width: 12ch;"
    return "width: 14ch; min-width: 14ch;"


def table_cell_style(column: str, *, header: bool = False) -> str:
    styles = [
        "padding: 0.4rem 0.6rem",
        "border: 1px solid #d0d7de",
        "text-align: right",
        "vertical-align: top",
        "overflow-wrap: anywhere",
        "word-break: break-word",
        "line-height: 1.35",
    ]
    if header:
        styles.extend(
            [
                "font-weight: 600",
                "background: #e9eef5",
                "color: #1f2328",
                "white-space: normal",
            ]
        )
    elif column in {"id", "case_study", "pair", "version", "time", "group", "table"} or column.endswith("_enabled"):
        styles.append("white-space: nowrap")
    return "; ".join(styles)


def format_table_value(column: str, value: object) -> str:
    if value is None or value == "":
        return ""
    if column == "case_study":
        return compact_case_label(value)
    return str(value)


def render_table(rows: list[dict[str, object]], columns: list[str]) -> HTML:
    colgroup = "".join(f"<col style=\"{table_column_style(column)}\">" for column in columns)
    header = "".join(
        f"<th style=\"{table_cell_style(column, header=True)}\">{escape(column)}</th>"
        for column in columns
    )
    body_rows = []
    for row in rows:
        body_cells = "".join(
            f"<td style=\"{table_cell_style(column)}\">{escape(format_table_value(column, row.get(column, '')))}</td>"
            for column in columns
        )
        body_rows.append(f"<tr>{body_cells}</tr>")
    empty_row = (
        f"<tr><td colspan='{len(columns)}' style=\"{table_cell_style('description')}\">No rows</td></tr>"
    )
    body = "".join(body_rows) or empty_row
    return HTML(
        "<div style=\"overflow-x: auto; margin: 0.5rem 0 1rem;\">"
        "<table style=\"border-collapse: collapse; table-layout: fixed; width: 100%;\">"
        f"<colgroup>{colgroup}</colgroup>"
        f"<thead><tr>{header}</tr></thead>"
        f"<tbody>{body}</tbody>"
        "</table>"
        "</div>"
    )


def load_plotly():
    try:
        go_module = import_module("plotly.graph_objects")
        make_subplots_fn = import_module("plotly.subplots").make_subplots
        return go_module, make_subplots_fn, None
    except ModuleNotFoundError as error:
        return None, None, error


def compact_case_label(case_or_case_study: object) -> str:
    case_study = getattr(case_or_case_study, "case_study", str(case_or_case_study))
    parts = case_study.split("_")
    if len(parts) >= 2 and parts[0] == "pair" and parts[1].isdigit():
        return f"pair{int(parts[1]):02d}"
    return case_study.replace("_", "")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
DATA_ROOT = REPO_ROOT / "data"
CODE_ROOT = REPO_ROOT / "code"

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from restaurant_simulation import discover_case_studies, run_case_pair

case_studies = discover_case_studies(data_root=DATA_ROOT)
case_lookup = {case.case_study: case for case in case_studies}

print(f"Repository root: {REPO_ROOT}")
print(f"Discovered {len(case_studies)} official case studies.")
print("This notebook is configured to run all official case studies with Run All.")


Repository root: /Users/grantwright/Desktop/COMP 1110/COMP-1110-C08
Discovered 7 official case studies.
This notebook is configured to run all official case studies with Run All.


## Run All Scope

This notebook now runs the full official case-study set by default. Use Jupyter's Run All to execute every official pair in one pass.


In [14]:
selected_cases = case_studies
selected_case_label = "All official case studies"

print(f"Notebook scope: {selected_case_label}")
print(f"Run All will execute {len(selected_cases)} pair comparison(s).")
print("Pairs in this run:")
for case in selected_cases:
    print(f"- {compact_case_label(case)}: {case.title}")


Notebook scope: All official case studies
Run All will execute 7 pair comparison(s).
Pairs in this run:
- pair01: Pair 01: Dessert Lounge vs Family Trattoria
- pair02: Pair 02: Single Queue vs Multi Queue
- pair03: Pair 03: Coarse vs Fine Queue Buckets
- pair04: Pair 04: No Reservation Hold vs Reservation Hold
- pair05: Pair 05: Low vs High Cleaning Capacity
- pair06: Pair 06: No Abandonment vs Abandonment Enabled
- pair07: Pair 07: FCFS vs Best-Fit Seating


## Scenario Execution

This section runs the full official case-study set, then materializes the main metrics and delta tables used by the rest of the notebook.


In [15]:
comparisons = [run_case_pair(case.case_study, data_root=DATA_ROOT) for case in selected_cases]

metrics_rows = []
delta_rows = []
for comparison in comparisons:
    case = case_lookup[comparison["case_study"]]
    for version in ["A", "B"]:
        metrics_rows.append(
            {
                "case_study": comparison["case_study"],
                "title": case.title,
                "version": version,
                **comparison[version]["metrics"],
            }
        )
    delta_rows.append(
        {
            "case_study": comparison["case_study"],
            "title": case.title,
            **comparison["metric_deltas_b_minus_a"],
        }
    )

print(f"Selection scope: {selected_case_label}")
print(f"Ran {len(comparisons)} pair comparison(s).")
print("Per-version metrics")
display(
    render_table(
        metrics_rows,
        [
            "case_study",
            "version",
            "average_wait_time",
            "groups_served",
            "groups_abandoned",
            "table_utilization_overall",
        ],
    )
)
print("Metric deltas (B minus A)")
display(
    render_table(
        delta_rows,
        [
            "case_study",
            "average_wait_time",
            "groups_served",
            "groups_abandoned",
            "table_utilization_overall",
        ],
    )
)


Selection scope: All official case studies
Ran 7 pair comparison(s).
Per-version metrics


case_study,version,average_wait_time,groups_served,groups_abandoned,table_utilization_overall
pair01,A,174.4444,36,0,0.1672
pair01,B,33.2222,36,0,0.1931
pair02,A,49.3214,28,0,0.1002
pair02,B,19.4286,28,0,0.1002
pair03,A,11.9722,36,0,0.2072
pair03,B,10.1667,36,0,0.2072
pair04,A,30.7419,31,0,0.3057
pair04,B,31.1935,31,0,0.3051
pair05,A,14.3667,30,0,0.1814
pair05,B,12.8,30,0,0.1713


Metric deltas (B minus A)


case_study,average_wait_time,groups_served,groups_abandoned,table_utilization_overall
pair01,-141.2222,0.0,0.0,0.0259
pair02,-29.8928,0.0,0.0,0.0
pair03,-1.8055,0.0,0.0,0.0
pair04,0.4516,0.0,0.0,-0.0006
pair05,-1.5667,0.0,0.0,-0.0101
pair06,-69.9643,-10.0,10.0,-0.029
pair07,-0.3667,0.0,0.0,0.0


## Headline Metric Comparison

This section compares the most important service outcomes across the current selection. If Plotly is available, it also renders grouped charts; otherwise the summary table remains readable on its own.


In [16]:
summary_metrics = [
    "average_wait_time",
    "groups_served",
    "groups_abandoned",
    "table_utilization_overall",
]

summary_rows = [
    {
        "title": row["title"],
        "version": row["version"],
        **{metric: row[metric] for metric in summary_metrics},
    }
    for row in metrics_rows
]

display(render_table(summary_rows, ["title", "version", *summary_metrics]))

go, make_subplots, plotly_error = load_plotly()
if go is None:
    print(f"Plotly is not available in this kernel ({plotly_error}). The summary table above still contains the core comparison results.")
else:
    summary_chart = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=tuple(metric.replace("_", " ").title() for metric in summary_metrics),
    )

    positions = [(1, 1), (1, 2), (2, 1), (2, 2)]
    case_labels = [compact_case_label(case) for case in selected_cases]
    for metric, (row_idx, col_idx) in zip(summary_metrics, positions):
        values_a = []
        values_b = []
        for case in selected_cases:
            option_a = next(row for row in metrics_rows if row["case_study"] == case.case_study and row["version"] == "A")
            option_b = next(row for row in metrics_rows if row["case_study"] == case.case_study and row["version"] == "B")
            values_a.append(option_a[metric])
            values_b.append(option_b[metric])
        summary_chart.add_trace(
            go.Bar(
                name=f"A {metric}",
                x=case_labels,
                y=values_a,
                legendgroup="A",
                showlegend=metric == summary_metrics[0],
            ),
            row=row_idx,
            col=col_idx,
        )
        summary_chart.add_trace(
            go.Bar(
                name=f"B {metric}",
                x=case_labels,
                y=values_b,
                legendgroup="B",
                showlegend=metric == summary_metrics[0],
            ),
            row=row_idx,
            col=col_idx,
        )

    summary_chart.update_layout(
        height=820,
        width=980,
        barmode="group",
        bargap=0.34,
        bargroupgap=0.12,
        title=f"Headline metrics for {selected_case_label}",
    )
    summary_chart.show()


title,version,average_wait_time,groups_served,groups_abandoned,table_utilization_overall
Pair 01: Dessert Lounge vs Family Trattoria,A,174.4444,36,0,0.1672
Pair 01: Dessert Lounge vs Family Trattoria,B,33.2222,36,0,0.1931
Pair 02: Single Queue vs Multi Queue,A,49.3214,28,0,0.1002
Pair 02: Single Queue vs Multi Queue,B,19.4286,28,0,0.1002
Pair 03: Coarse vs Fine Queue Buckets,A,11.9722,36,0,0.2072
Pair 03: Coarse vs Fine Queue Buckets,B,10.1667,36,0,0.2072
Pair 04: No Reservation Hold vs Reservation Hold,A,30.7419,31,0,0.3057
Pair 04: No Reservation Hold vs Reservation Hold,B,31.1935,31,0,0.3051
Pair 05: Low vs High Cleaning Capacity,A,14.3667,30,0,0.1814
Pair 05: Low vs High Cleaning Capacity,B,12.8,30,0,0.1713


## Delta Analysis

This section focuses on the change from Option A to Option B, helping the reader understand what actually improves, worsens, or stays flat after switching policies.


In [17]:
interesting_delta_metrics = [
    "average_wait_time",
    "groups_served",
    "groups_abandoned",
    "table_utilization_overall",
]

display(render_table(delta_rows, ["title", *interesting_delta_metrics]))

go, _, plotly_error = load_plotly()
if go is None:
    print(f"Plotly is not available in this kernel ({plotly_error}). The delta table above still shows the B minus A changes.")
else:
    delta_chart = go.Figure()
    case_labels = [compact_case_label(row["case_study"]) for row in delta_rows]
    for metric in interesting_delta_metrics:
        delta_chart.add_trace(
            go.Bar(
                name=metric.replace("_", " ").title(),
                x=case_labels,
                y=[row[metric] for row in delta_rows],
            )
        )
    delta_chart.add_hline(y=0)
    delta_chart.update_layout(
        title=f"Option B minus Option A for {selected_case_label}",
        barmode="group",
        width=920,
        height=500,
        bargap=0.34,
        bargroupgap=0.12,
    )
    delta_chart.show()


title,average_wait_time,groups_served,groups_abandoned,table_utilization_overall
Pair 01: Dessert Lounge vs Family Trattoria,-141.2222,0.0,0.0,0.0259
Pair 02: Single Queue vs Multi Queue,-29.8928,0.0,0.0,0.0
Pair 03: Coarse vs Fine Queue Buckets,-1.8055,0.0,0.0,0.0
Pair 04: No Reservation Hold vs Reservation Hold,0.4516,0.0,0.0,-0.0006
Pair 05: Low vs High Cleaning Capacity,-1.5667,0.0,0.0,-0.0101
Pair 06: No Abandonment vs Abandonment Enabled,-69.9643,-10.0,10.0,-0.029
Pair 07: FCFS vs Best-Fit Seating,-0.3667,0.0,0.0,0.0


## Pair Deep Dive

This section drills into each selected pair with per-case metric tables, outcome breakdowns, and queue summaries, plus charts when optional plotting support is available.


In [18]:
headline_metrics = [
    "average_wait_time",
    "max_queue_length",
    "groups_served",
    "groups_abandoned",
]

go, make_subplots, plotly_error = load_plotly()

for comparison in comparisons:
    case = case_lookup[comparison["case_study"]]

    metric_rows = [
        {
            "metric": metric,
            "Option A": comparison["A"]["metrics"][metric],
            "Option B": comparison["B"]["metrics"][metric],
        }
        for metric in headline_metrics
    ]

    outcome_counts = {
        version: Counter(outcome["status"] for outcome in comparison[version]["group_outcomes"])
        for version in ["A", "B"]
    }
    outcome_statuses = sorted({status for counter in outcome_counts.values() for status in counter})
    outcome_rows = [
        {
            "version": version,
            **{status: outcome_counts[version].get(status, 0) for status in outcome_statuses},
        }
        for version in ["A", "B"]
    ]

    queue_rows = [
        {
            "version": version,
            "minute": snapshot["minute"],
            "total_waiting": snapshot["total_waiting"],
        }
        for version in ["A", "B"]
        for snapshot in comparison[version]["queue_snapshots"]
    ]
    queue_summary_rows = []
    for version in ["A", "B"]:
        version_rows = [row for row in queue_rows if row["version"] == version]
        queue_summary_rows.append(
            {
                "version": version,
                "peak_waiting": max((row["total_waiting"] for row in version_rows), default=0),
                "final_waiting": version_rows[-1]["total_waiting"] if version_rows else 0,
                "snapshot_count": len(version_rows),
            }
        )

    print(case.title)
    display(render_table(metric_rows, ["metric", "Option A", "Option B"]))
    display(render_table(outcome_rows, ["version", *outcome_statuses]))
    display(render_table(queue_summary_rows, ["version", "peak_waiting", "final_waiting", "snapshot_count"]))

    if go is None:
        print(f"Plotly is not available in this kernel ({plotly_error}). The tables above provide the deep-dive summary for this pair.")
        continue

    subplot = make_subplots(
        rows=1,
        cols=3,
        subplot_titles=(
            "Headline metrics",
            "Group outcomes",
            "Queue pressure over time",
        ),
        column_widths=(0.42, 0.18, 0.40),
        horizontal_spacing=0.08,
    )

    subplot.add_trace(
        go.Bar(name="Option A", x=[row["metric"] for row in metric_rows], y=[row["Option A"] for row in metric_rows]),
        row=1,
        col=1,
    )
    subplot.add_trace(
        go.Bar(name="Option B", x=[row["metric"] for row in metric_rows], y=[row["Option B"] for row in metric_rows]),
        row=1,
        col=1,
    )

    for status in outcome_statuses:
        subplot.add_trace(
            go.Bar(
                name=f"status: {status}",
                x=["A", "B"],
                y=[outcome_counts["A"].get(status, 0), outcome_counts["B"].get(status, 0)],
            ),
            row=1,
            col=2,
        )

    for version in ["A", "B"]:
        version_rows = [row for row in queue_rows if row["version"] == version]
        subplot.add_trace(
            go.Scatter(
                x=[row["minute"] for row in version_rows],
                y=[row["total_waiting"] for row in version_rows],
                mode="lines+markers",
                name=f"Queue {version}",
            ),
            row=1,
            col=3,
        )

    subplot.update_layout(
        title=compact_case_label(case),
        barmode="group",
        width=1140,
        height=450,
        bargap=0.32,
        bargroupgap=0.14,
    )
    subplot.update_xaxes(title_text="Metric", row=1, col=1)
    subplot.update_xaxes(title_text="Version", row=1, col=2)
    subplot.update_xaxes(title_text="Minute of service", row=1, col=3)
    subplot.update_yaxes(title_text="Value", row=1, col=1)
    subplot.update_yaxes(title_text="Parties", row=1, col=2)
    subplot.update_yaxes(title_text="Waiting parties", row=1, col=3)
    subplot.show()


Pair 01: Dessert Lounge vs Family Trattoria


metric,Option A,Option B
average_wait_time,174.4444,33.2222
max_queue_length,19,11
groups_served,36,36
groups_abandoned,0,0


version,completed
A,36
B,36


version,peak_waiting,final_waiting,snapshot_count
A,18,0,100
B,11,0,91


Pair 02: Single Queue vs Multi Queue


...

version,peak_waiting,final_waiting,snapshot_count
A,11,0,75
B,5,0,70


Pair 07: FCFS vs Best-Fit Seating


metric,Option A,Option B
average_wait_time,14.0,13.6333
max_queue_length,10,10
groups_served,30,30
groups_abandoned,0,0


version,completed
A,30
B,30


version,peak_waiting,final_waiting,snapshot_count
A,10,0,75
B,10,0,75
